In [8]:
# Cell 1: Read all PNG images and print unique pixel values with color swatches
import os
import numpy as np
from PIL import Image
import glob
from IPython.display import display, HTML

# ── Configure your input directory here ──────────────────────────────────────
INPUT_DIR = '/home/eric/Desktop/newdataset/relabeled'   # <-- change this
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.isdir(INPUT_DIR):
    raise NotADirectoryError(f'Directory not found: {INPUT_DIR!r}')

png_files = sorted(glob.glob(os.path.join(INPUT_DIR, '*.png')))

if not png_files:
    raise FileNotFoundError(f'No PNG files found in {INPUT_DIR!r}')

print(f'Found {len(png_files)} PNG file(s):')
for f in png_files:
    print(f'  - {f}')

# Load all images as grayscale arrays
images = []
for f in png_files:
    arr = np.array(Image.open(f).convert('L'))
    images.append((f, arr))

# Unique pixel values from the FIRST image
first_filename, first_array = images[0]
total_pixels = first_array.size
unique_values, counts = np.unique(first_array, return_counts=True)

print(f'\nFirst image : {first_filename}')
print(f'Shape       : {first_array.shape}  ({total_pixels:,} pixels)')
print(f'Unique pixel values: {len(unique_values)}')

# Build HTML table with inline color swatches
rows = ''
for pv, count in zip(unique_values, counts):
    v = int(pv)
    pct = count / total_pixels * 100
    rgb_str = f'({v}, {v}, {v})'
    swatch = (
        f'<span style="'
        f'display:inline-block;'
        f'width:18px;height:18px;'
        f'background-color:rgb({v},{v},{v});'
        f'border:1px solid #aaa;'
        f'vertical-align:middle;'
        f'margin-right:6px;'
        f'"></span>'
    )
    rows += (
        f'<tr>'
        f'<td style="text-align:center">{swatch}</td>'
        f'<td style="text-align:right;padding:4px 12px">{v}</td>'
        f'<td style="text-align:center;padding:4px 12px">{rgb_str}</td>'
        f'<td style="text-align:right;padding:4px 12px">{count:,}</td>'
        f'<td style="text-align:right;padding:4px 12px">{pct:.2f}%</td>'
        f'</tr>'
    )

html = f'''
<table style="border-collapse:collapse;font-family:monospace;font-size:13px">
  <thead>
    <tr style="border-bottom:2px solid #ccc">
      <th style="padding:4px 12px">Color</th>
      <th style="padding:4px 12px">Grayscale</th>
      <th style="padding:4px 12px">RGB</th>
      <th style="padding:4px 12px">Count</th>
      <th style="padding:4px 12px">% of pixels</th>
    </tr>
  </thead>
  <tbody>{rows}</tbody>
</table>
'''
display(HTML(html))

Found 11 PNG file(s):
  - /home/eric/Desktop/newdataset/relabeled/N6-1.png
  - /home/eric/Desktop/newdataset/relabeled/N6-10.png
  - /home/eric/Desktop/newdataset/relabeled/N6-11.png
  - /home/eric/Desktop/newdataset/relabeled/N6-2.png
  - /home/eric/Desktop/newdataset/relabeled/N6-3.png
  - /home/eric/Desktop/newdataset/relabeled/N6-4.png
  - /home/eric/Desktop/newdataset/relabeled/N6-5.png
  - /home/eric/Desktop/newdataset/relabeled/N6-6.png
  - /home/eric/Desktop/newdataset/relabeled/N6-7.png
  - /home/eric/Desktop/newdataset/relabeled/N6-8.png
  - /home/eric/Desktop/newdataset/relabeled/N6-9.png

First image : /home/eric/Desktop/newdataset/relabeled/N6-1.png
Shape       : (256, 256)  (65,536 pixels)
Unique pixel values: 11


Color,Grayscale,RGB,Count,% of pixels
,0,"(0, 0, 0)","60,533",92.37%
,1,"(1, 1, 1)",711,1.08%
,2,"(2, 2, 2)",469,0.72%
,3,"(3, 3, 3)",513,0.78%
,4,"(4, 4, 4)",414,0.63%
,5,"(5, 5, 5)",452,0.69%
,6,"(6, 6, 6)",436,0.67%
,7,"(7, 7, 7)",464,0.71%
,8,"(8, 8, 8)",453,0.69%
,9,"(9, 9, 9)",662,1.01%


In [6]:
# Cell 2: Custom pixel value mapping (0-255 grayscale) and save all images to 'relabeled/'
#
# Edit REPLACE_MAP using the pixel values printed in Cell 1.
# Format: old_pixel_value: new_pixel_value  (both 0-255)
# Any pixel value NOT listed here will be kept as-is.

REPLACE_MAP = {
    255:   0,   # <-- edit as needed
    35:    0,   # <-- edit as needed
    33:  1,   # <-- edit as needed
    15:  2,   # <-- edit as needed
    11:  3,   # <-- edit as needed
    28:  4,   # <-- edit as needed
    26:  5,   # <-- edit as needed
    29:  6,   # <-- edit as needed
    5:  7,   # <-- edit as needed
    13:  8,   # <-- edit as needed
    6:  9,   # <-- edit as needed
    24:  10,   # <-- edit as needed
    # add more rows as needed...
}

print('Custom pixel value replacement map:')
print(f'  {"Old value":>10}  ->  {"New value"}')
print('-' * 28)
for old, new in REPLACE_MAP.items():
    print(f'  {old:>10}  ->  {new}')
print()

output_dir = os.path.join(INPUT_DIR, 'relabeled')
os.makedirs(output_dir, exist_ok=True)

for filename, arr in images:
    modified = arr.copy()
    for old_val, new_val in REPLACE_MAP.items():
        mask = arr == old_val
        modified[mask] = new_val
        changed = int(mask.sum())
        if changed:
            print(f'  [{os.path.basename(filename)}] {old_val} -> {new_val}: {changed:,} pixel(s)')

    out_path = os.path.join(output_dir, os.path.basename(filename))
    Image.fromarray(modified.astype(np.uint8)).save(out_path)
    print(f'  Saved -> {out_path}')

print(f'\nDone. All {len(images)} image(s) written to {output_dir}/')

Custom pixel value replacement map:
   Old value  ->  New value
----------------------------
         255  ->  0
          35  ->  0
          33  ->  1
          15  ->  2
          11  ->  3
          28  ->  4
          26  ->  5
          29  ->  6
           5  ->  7
          13  ->  8
           6  ->  9
          24  ->  10

  [N6-10_watershed_mask.png] 255 -> 0: 1,020 pixel(s)
  [N6-10_watershed_mask.png] 35 -> 0: 60,775 pixel(s)
  [N6-10_watershed_mask.png] 33 -> 1: 581 pixel(s)
  [N6-10_watershed_mask.png] 15 -> 2: 351 pixel(s)
  [N6-10_watershed_mask.png] 11 -> 3: 445 pixel(s)
  [N6-10_watershed_mask.png] 28 -> 4: 242 pixel(s)
  [N6-10_watershed_mask.png] 26 -> 5: 385 pixel(s)
  [N6-10_watershed_mask.png] 29 -> 6: 293 pixel(s)
  [N6-10_watershed_mask.png] 5 -> 7: 418 pixel(s)
  [N6-10_watershed_mask.png] 13 -> 8: 275 pixel(s)
  [N6-10_watershed_mask.png] 6 -> 9: 436 pixel(s)
  [N6-10_watershed_mask.png] 24 -> 10: 315 pixel(s)
  Saved -> /home/eric/Desktop/newdataset/relabel